# NB2 — Attention Visualization

In NB1 you pulled attention patterns out as `[head, dest, src]` tensors and stared at raw numbers.
That doesn't scale — GPT-2 small has **144 heads**. This notebook is about *seeing* them.

We'll use **`circuitsvis`**, the standard interactive-visualization companion to TransformerLens, to
render attention patterns, and we'll learn to recognize a few **canonical head shapes** by eye:

- **current-token heads** — attend to *themselves* (bright main diagonal),
- **previous-token heads** — attend to the token one position *back* (bright sub-diagonal),
- **duplicate-token heads** — attend to an *earlier copy* of the current token (we'll only foreshadow
  these here; they need repeated text, which is NB3's whole setup).

By the end you'll be able to point at a head and say what it's doing — the skill the entire induction
hunt rests on.

## 0. Setup

Same model setup as before, plus `circuitsvis` (imported as `cv`). If the visualizations don't render
in your environment, make sure the notebook is trusted / running in a real Jupyter frontend.

In [ ]:
import torch
import circuitsvis as cv
from transformer_lens import HookedTransformer

torch.set_grad_enabled(False)
device = "cuda" if torch.cuda.is_available() else "cpu"
model = HookedTransformer.from_pretrained("gpt2", device=device)
print("circuitsvis", cv.__version__, "| model on", device)

## 1. Cache a sentence

We need two things to visualize a layer: the **string tokens** (for axis labels) and the layer's
**attention pattern** `[head, dest, src]`. We use a sentence with a repeated name (`John`, `Mary`)
so later we can look for heads that connect the repeats.

In [ ]:
text = "When Mary and John went to the store, John gave a bottle of milk to Mary because she was thirsty."
tokens = model.to_tokens(text)
str_tokens = model.to_str_tokens(text)   # list of strings, one per position (axis labels)

logits, cache = model.run_with_cache(tokens)
print("seq length:", len(str_tokens))
print(str_tokens)

## 2. `circuitsvis.attention.attention_patterns`

The core call is:

```python
cv.attention.attention_patterns(tokens=str_tokens, attention=pattern)
```

where `pattern` is `[n_heads, dest, src]` — i.e. `cache["pattern", layer][0]` (we drop the batch
dim). It renders **all heads in the layer** with a head selector.

**How to read it:** it's the `[dest, src]` matrix from NB1, drawn as a grid. Each **row is a
destination** (query) token; the bright cells in that row are the **source** (key) tokens it attends
to. Because attention is causal, everything is on or below the diagonal.

Let's start with **layer 0**. Look at **head 1** — you should see an almost pure **main diagonal**:
every token attends to *itself*. That's a **current-token head**.

In [ ]:
# All 12 heads of layer 0. Use the head selector; hover cells to read weights.
cv.attention.attention_patterns(tokens=str_tokens, attention=cache["pattern", 0][0])

## 3. A previous-token head

Now **layer 4**. Look at **head 11**: instead of the main diagonal, the bright line sits **one step
below** it — every token attends to the token *immediately before* it. That's a **previous-token
head**, and L4H11 is a famously clean one in GPT-2 small.

Keep this head in mind: previous-token heads are the **first half** of the induction circuit. In NB4
we'll see how a head like this feeds an induction head in a later layer.

In [ ]:
# All 12 heads of layer 4 — select head 11 to see the sub-diagonal previous-token pattern.
cv.attention.attention_patterns(tokens=str_tokens, attention=cache["pattern", 4][0])

## 4. From eyeballs to numbers: the diagonal trick

Reading grids works but doesn't scale to 144 heads. We can turn each canonical shape into a **single
score** using tensor diagonals:

- **current-token score** = mean of the **main diagonal** of `pattern` (`offset=0`): how much each
  token attends to itself, averaged over positions.
- **previous-token score** = mean of the **first sub-diagonal** (`offset=-1`): how much each token
  attends to the one before it.

`tensor.diagonal(offset=k, dim1=-2, dim2=-1)` pulls out diagonal `k` of the last two dims. Here's the
current-token score for every head, with the top few printed — L0H1 should dominate.

In [ ]:
def current_token_score(cache, model):
    """Mean main-diagonal attention for every head -> [n_layers, n_heads]."""
    nL, nH = model.cfg.n_layers, model.cfg.n_heads
    scores = torch.zeros(nL, nH)
    for L in range(nL):
        p = cache["pattern", L][0]                      # [head, dest, src]
        scores[L] = p.diagonal(dim1=-2, dim2=-1).mean(-1).cpu()   # mean over positions
    return scores

cur = current_token_score(cache, model)

# Print the top-5 heads as (layer, head, score).
flat = cur.flatten()
for i in flat.topk(5).indices:
    L, H = int(i) // model.cfg.n_heads, int(i) % model.cfg.n_heads
    print(f"L{L}H{H}: current-token score = {flat[i]:.3f}")

## Canonical head shapes — reference

| Shape | Where the bright cells are | `diagonal` offset | Example (GPT-2 small) |
|-------|---------------------------|-------------------|-----------------------|
| current-token | main diagonal (attend to self) | `0` | **L0H1** |
| previous-token | one below diagonal (attend to *i−1*) | `−1` | **L4H11** |
| duplicate-token | at an *earlier copy* of the current token | (needs repeats) | seen in NB3 |
| **induction** | at the token *after* the previous copy | (needs repeats) | the goal — NB3 |

Duplicate-token and induction heads only reveal themselves when the input **repeats**, because they
act on earlier occurrences of the current token. That's exactly why NB3 feeds the model
repeated-random-token sequences.

## Practice

Mirror the `current_token_score` function to build a **`previous_token_score`** and use it to find the
cleanest previous-token head in GPT-2 small.

Task:
1. Write `previous_token_score(cache, model)` returning a `[n_layers, n_heads]` tensor — the same as
   `current_token_score` but using the **first sub-diagonal** (`offset=-1`) instead of the main one.
2. Print the **top-5** heads by that score.
3. Confirm **L4H11** comes out on top, then open the layer-4 visualization above and verify head 11
   really is the sub-diagonal one.

Hint: the only change from `current_token_score` is `p.diagonal(offset=-1, dim1=-2, dim2=-1)`.

In [ ]:
# TODO(you):
# def previous_token_score(cache, model):
#     ...
#     use p.diagonal(offset=-1, dim1=-2, dim2=-1).mean(-1)
#
# prev = previous_token_score(cache, model)
# print the top-5 (layer, head, score) -- expect L4H11 first


<details>
<summary>Reference solution (open after you've tried)</summary>

```python
def previous_token_score(cache, model):
    nL, nH = model.cfg.n_layers, model.cfg.n_heads
    scores = torch.zeros(nL, nH)
    for L in range(nL):
        p = cache["pattern", L][0]                                    # [head, dest, src]
        scores[L] = p.diagonal(offset=-1, dim1=-2, dim2=-1).mean(-1).cpu()
    return scores

prev = previous_token_score(cache, model)
flat = prev.flatten()
for i in flat.topk(5).indices:
    L, H = int(i) // model.cfg.n_heads, int(i) % model.cfg.n_heads
    print(f"L{L}H{H}: previous-token score = {flat[i]:.3f}")
```

Expected top: `L4H11` with a score near 1.0.

</details>

---
**Done?** Once your `previous_token_score` puts L4H11 on top and you've matched it to the picture,
ask me to review. Next is **NB3 — induction detection**, where we build repeated-random-token
sequences, watch the loss drop on the second copy, and write an **induction score** that finally
pinpoints the induction heads among all 144.